In [1]:
from gen_catalyst_design.discrete_space_diffusion import DiffusionModel
from gen_catalyst_design.discrete_space_diffusion.Dataset import get_dataloaders_from_atoms_list
from ase.db import connect
from ase_ml_models.databases import get_atoms_list_from_db
from ase.io import read
import torch.nn.functional as F
import torch
from ase.io import write

/opt/anaconda3/envs/cat_opt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
use_absorbing_state = True
mask_classes = True
miller_index = "100"
element_pool = ["Au","Cu","Pd","Rh","Ni","Ga"]
if use_absorbing_state:
    element_pool = ["(X)"] + element_pool

atoms_list = read("dataset.traj", index=":")

diff_model = DiffusionModel.load_from_checkpoint("model_001/checkpoints/last.ckpt")

random_seed = 42
torch.manual_seed(random_seed)
torch.cuda.manual_seed_all(random_seed)
train_loader, val_loader = get_dataloaders_from_atoms_list(
        atoms_list=atoms_list,
        element_pool=element_pool,
        batch_size=40,
    )
for i, batch in enumerate(train_loader):
    diff_model.calculate_loss_terms(batch=batch, batch_idx=i)
    break